# Use cases: groups, roles and datasets end to end

The other notebooks walk the API one call at a time ([groups_example](groups_example.ipynb),
[datasets_example](datasets_example.ipynb)). This one works the other way around: it takes real
workflows and shows which calls carry them. The running example is a studio that shoots garments
twice - once on a hanger, once on a model - and trains a try-on generation model on the pairs.

| Use case | What it shows |
|---|---|
| [1. Paired training data](#3.-Use-case-1:-paired-training-data) | one group per garment, roles pair the shots, a frozen dataset feeds the trainer |
| [2. Train/eval split](#4.-Use-case-2:-a-train/eval-split) | split into two disjoint datasets |
| [3. The next drop](#5.-Use-case-3:-the-next-drop) | new shoots arrive, the frozen v1 stays put, v2 = copy + add |
| [4. What is still missing](#6.-Use-case-4:-what-is-still-missing) | role queries as a QA queue, image-side export of a whole dataset |
| [5. A second taxonomy](#7.-Use-case-5:-a-second-taxonomy) | none of the role or type names are built in - another team, other names |

## 1. Connect

In [1]:
import os
import time
import uuid

from dataroom_client import DataRoomClientSync, DataRoomError

os.environ["DATAROOM_API_KEY"] = 'YOUR_KEY_HERE'
os.environ["DATAROOM_API_URL"] = 'http://localhost:8000/api/'

client = DataRoomClientSync()
try:
    client.images.list(limit=1)
except DataRoomError as e:
    raise RuntimeError('API token rejected. Check DATAROOM_API_KEY.') from e

# Everything this notebook creates carries this suffix, so re-runs never collide.
RUN = uuid.uuid4().hex[:6]
print('connected; run id:', RUN)


def why(exc):
    """The server's validation message, not httpx's status line."""
    response = getattr(exc, 'response', None)
    return response.text if response is not None else str(exc)

connected; run id: b249d6


## 2. The team's taxonomy

Roles and GroupTypes are defined by you, per instance - the server ships with none. The try-on
team needs three ways a garment can be photographed and one group shape that ties them together:

| | |
|---|---|
| `on_hang` | the input: garment on a hanger, no model |
| `on_model_front` | the target: the same garment worn, shot from the front |
| `detail_shot` | optional close-up, used by QA in use case 4 |

A `garment_pair` group holds one garment's shots. The two roles the trainer needs are required,
so an incomplete pair cannot even be created - validation runs on write, not in the training job.

In [2]:
ROLES = {
    'on_hang': 'Garment on a hanger or mannequin, no model.',
    'on_model_front': 'The same garment worn by a model, shot from the front.',
    'detail_shot': 'Optional close-up of fabric or print.',
}
for name, description in ROLES.items():
    try:
        client.roles.create(name=name, description=description)
        print('created role', name)
    except DataRoomError:
        print('role exists', name)

try:
    client.group_types.create(
        name='garment_pair',
        description='One garment: the hanger shot paired with the on-model shot.',
        metadata_schema={'type': 'object', 'additionalProperties': True},
        roles=[
            {'role': 'on_hang',        'is_required': True},
            {'role': 'on_model_front', 'is_required': True},
            {'role': 'detail_shot',    'is_required': False},
        ],
    )
    print('created group_type garment_pair')
except DataRoomError:
    print('group_type exists garment_pair')

role exists on_hang
role exists on_model_front
role exists detail_shot


group_type exists garment_pair


## 3. Use case 1: paired training data

A shoot day produces shots for four garments. One `groups.create_many` call turns them into
four groups - the role names do the pairing, and the required roles guarantee no group is
missing half its pair. The first two garments also got a close-up.

In [3]:
image_ids = []
for img in client.images.list(limit=14, fields=['id']):
    image_ids.append(img['id'])
assert len(image_ids) >= 14, 'need at least 14 images - run import_images first'

groups = client.groups.create_many([
    {'name': f'garment-{RUN}-01', 'type': 'garment_pair', 'metadata': {'sku': f'{RUN}-01'}, 'members': [
        {'image_id': image_ids[0], 'role': 'on_hang'},
        {'image_id': image_ids[1], 'role': 'on_model_front'},
        {'image_id': image_ids[2], 'role': 'detail_shot'},
    ]},
    {'name': f'garment-{RUN}-02', 'type': 'garment_pair', 'metadata': {'sku': f'{RUN}-02'}, 'members': [
        {'image_id': image_ids[3], 'role': 'on_hang'},
        {'image_id': image_ids[4], 'role': 'on_model_front'},
        {'image_id': image_ids[5], 'role': 'detail_shot'},
    ]},
    {'name': f'garment-{RUN}-03', 'type': 'garment_pair', 'metadata': {'sku': f'{RUN}-03'}, 'members': [
        {'image_id': image_ids[6], 'role': 'on_hang'},
        {'image_id': image_ids[7], 'role': 'on_model_front'},
    ]},
    {'name': f'garment-{RUN}-04', 'type': 'garment_pair', 'metadata': {'sku': f'{RUN}-04'}, 'members': [
        {'image_id': image_ids[8], 'role': 'on_hang'},
        {'image_id': image_ids[9], 'role': 'on_model_front'},
    ]},
])

group_ids = []
for g in groups:
    group_ids.append(g['id'])
    print(g['name'], '-', g['image_count'], 'images')

garment-b249d6-03 - 2 images
garment-b249d6-04 - 2 images
garment-b249d6-01 - 3 images
garment-b249d6-02 - 3 images


### Collect them into a dataset and freeze it

The dataset is the unit the trainer consumes: `slug/version`, one GroupType, freezable.
Freezing pins the membership - a training run that records `tryon-pairs-<run>/1` can be
reproduced long after the data keeps moving elsewhere.

In [4]:
SLUG = f'tryon-pairs-{RUN}'
ds = client.datasets.create(name='Try-on pairs', slug=SLUG, type='garment_pair',
                            description='Hanger/on-model pairs for try-on training.')
SV = ds['slug_version']

client.datasets.add_groups(SV, group_ids)
client.datasets.freeze(SV)
print(SV, '| group_count =', client.datasets.get(SV)['group_count'], '| frozen')

# Frozen means frozen - membership writes are rejected until unfreeze.
try:
    client.datasets.add_groups(SV, [group_ids[0]])
    print('BUG: a frozen dataset accepted new groups')
except DataRoomError as e:
    print('add_groups on frozen ->', why(e))

tryon-pairs-b249d6/1 | group_count = 4 | frozen
add_groups on frozen -> ["Dataset is frozen"]


### Read the pairs back for the trainer

One hydrated request per dataset: `datasets.groups` is `groups.list(dataset=...)` and takes the
same `include_*` flags. `include_presigned_urls` attaches a download URL per member, so the
dataloader below only pairs URLs by role - no image-by-image fetching.
(`include_thumbnail_urls` works the same way when the reduced size is enough.)

In [5]:
hydrated = client.datasets.groups(SV, include_presigned_urls=True)

for g in hydrated:
    hang_url = None
    model_url = None
    for member in g['roles']:
        if member['role'] == 'on_hang':
            hang_url = member['presigned_url']
        if member['role'] == 'on_model_front':
            model_url = member['presigned_url']
    print(g['name'])
    print('  input  =', hang_url[:70])
    print('  target =', model_url[:70])

garment-b249d6-04
  input  = http://localhost:9000/dataroom-local/images/bench_0_105/original.png
  target = http://localhost:9000/dataroom-local/images/bench_0_106/original.png
garment-b249d6-03
  input  = http://localhost:9000/dataroom-local/images/bench_0_103/original.png
  target = http://localhost:9000/dataroom-local/images/bench_0_104/original.png
garment-b249d6-02
  input  = http://localhost:9000/dataroom-local/images/bench_0_100/original.png
  target = http://localhost:9000/dataroom-local/images/bench_0_101/original.png
garment-b249d6-01
  input  = http://localhost:9000/dataroom-local/images/bench_0_0/original.png
  target = http://localhost:9000/dataroom-local/images/bench_0_1/original.png


## 4. Use case 2: a train/eval split

Two datasets over the same pool of groups: the first three go to train, the last one to eval.
A group is in one dataset or the other, never both - and both are frozen, so the split cannot
drift under a running experiment.

In [6]:
train_ids = group_ids[:3]
eval_ids = group_ids[3:]

train = client.datasets.create(name='Try-on train', slug=f'tryon-train-{RUN}', type='garment_pair')
TRAIN_SV = train['slug_version']
client.datasets.add_groups(TRAIN_SV, train_ids)
client.datasets.freeze(TRAIN_SV)
print(TRAIN_SV, '-', len(train_ids), 'groups, frozen')

evaluation = client.datasets.create(name='Try-on eval', slug=f'tryon-eval-{RUN}', type='garment_pair')
EVAL_SV = evaluation['slug_version']
client.datasets.add_groups(EVAL_SV, eval_ids)
client.datasets.freeze(EVAL_SV)
print(EVAL_SV, '-', len(eval_ids), 'groups, frozen')

overlap = 0
for g in client.datasets.groups(EVAL_SV):
    if g['id'] in train_ids:
        overlap = overlap + 1
print('groups in both datasets:', overlap)

tryon-train-b249d6/1 - 3 groups, frozen


tryon-eval-b249d6/1 - 1 groups, frozen
groups in both datasets: 0


## 5. Use case 3: the next drop

Two weeks later the studio delivers two more garments. The frozen v1 is mid-training and must
not move. `datasets.copy` to the same slug mints the next version with the same members; the new
groups then go into v2 only. Jobs pin `slug/version`, so nothing downstream shifts by surprise.

In [7]:
drop = client.groups.create_many([
    {'name': f'garment-{RUN}-d1', 'type': 'garment_pair', 'metadata': {'sku': f'{RUN}-d1'}, 'members': [
        {'image_id': image_ids[10], 'role': 'on_hang'},
        {'image_id': image_ids[11], 'role': 'on_model_front'},
    ]},
    {'name': f'garment-{RUN}-d2', 'type': 'garment_pair', 'metadata': {'sku': f'{RUN}-d2'}, 'members': [
        {'image_id': image_ids[12], 'role': 'on_hang'},
        {'image_id': image_ids[13], 'role': 'on_model_front'},
    ]},
])

drop_ids = []
for g in drop:
    drop_ids.append(g['id'])

v2 = client.datasets.copy(SV, name='Try-on pairs, second drop', slug=SLUG)
V2 = v2['slug_version']
client.datasets.add_groups(V2, drop_ids)

print('versions of', SLUG + ':')
for d in client.datasets.list(slug=SLUG):
    d = client.datasets.get(d['slug_version'])
    print(' ', d['slug_version'], '- groups =', d['group_count'], '- frozen =', d['is_frozen'])

versions of tryon-pairs-b249d6:
  tryon-pairs-b249d6/2 - groups = 6 - frozen = False
  tryon-pairs-b249d6/1 - groups = 4 - frozen = True


## 6. Use case 4: what is still missing

`detail_shot` is optional at write time, but QA wants every garment to end up with one.
`has_role` filters on actual memberships (vs `roles`, which filters on what the *type*
declares), so the difference between "all groups in the dataset" and "groups that have a
`detail_shot`" is exactly the annotation queue.

In [8]:
in_v2 = client.groups.list(dataset=V2)
with_detail = client.groups.list(dataset=V2, has_role=['detail_shot'])

names_with_detail = []
for g in with_detail:
    names_with_detail.append(g['name'])

missing = []
for g in in_v2:
    if g['name'] not in names_with_detail:
        missing.append(g['name'])

print(len(in_v2), 'groups in', V2, '-', len(with_detail), 'already have a detail_shot')
print('annotation queue:')
for name in sorted(missing):
    print(' ', name)

6 groups in tryon-pairs-b249d6/2 - 2 already have a detail_shot
annotation queue:
  garment-b249d6-03
  garment-b249d6-04
  garment-b249d6-d1
  garment-b249d6-d2


### The image side of a dataset

The membership chain (dataset -> group -> image) is denormalized onto each image, so images are
filterable by dataset like by any other field - here, every image v2 touches, e.g. for an export.
OpenSearch applies that denorm asynchronously, hence the retry loop.

In [9]:
expected = 0
for g in in_v2:
    expected = expected + g['image_count']

hits = client.images.list(datasets=[V2], fields=['id'], limit=200)
for attempt in range(60):
    if len(hits) >= expected:
        break
    time.sleep(0.5)
    hits = client.images.list(datasets=[V2], fields=['id'], limit=200)
print(len(hits), 'images reachable through', V2, '- expected', expected)

# Dataset filters compose with every other image filter - the eval inputs only:
eval_inputs = client.images.list(datasets=[EVAL_SV], roles=['on_hang'], fields=['id'])
print('on_hang images in', EVAL_SV, ':', len(eval_inputs))

14 images reachable through tryon-pairs-b249d6/2 - expected 14
on_hang images in tryon-eval-b249d6/1 : 2


## 7. Use case 5: a second taxonomy

Nothing about `garment_pair` is built in - another team on the same instance names its own
world. Interior staging pairs the empty room with the furnished shot; the flow is the same
three calls: roles, a type, groups into a dataset.

In [10]:
try:
    client.roles.create(name='empty_room', description='The room before staging - bare walls and floor.')
    print('created role empty_room')
except DataRoomError:
    print('role exists empty_room')
try:
    client.roles.create(name='staged_room', description='The same room furnished and dressed.')
    print('created role staged_room')
except DataRoomError:
    print('role exists staged_room')

try:
    client.group_types.create(
        name='room_scene',
        description='One room: empty and staged shots of the same space.',
        metadata_schema={'type': 'object', 'additionalProperties': True},
        roles=[
            {'role': 'empty_room',  'is_required': True},
            {'role': 'staged_room', 'is_required': True},
        ],
    )
    print('created group_type room_scene')
except DataRoomError:
    print('group_type exists room_scene')

# The same image can serve in groups of different types - membership never claims an image.
rooms = client.groups.create_many([
    {'name': f'room-{RUN}-1', 'type': 'room_scene', 'members': [
        {'image_id': image_ids[0], 'role': 'empty_room'},
        {'image_id': image_ids[1], 'role': 'staged_room'},
    ]},
    {'name': f'room-{RUN}-2', 'type': 'room_scene', 'members': [
        {'image_id': image_ids[2], 'role': 'empty_room'},
        {'image_id': image_ids[3], 'role': 'staged_room'},
    ]},
])

staging = client.datasets.create(name='Staging scenes', slug=f'staging-scenes-{RUN}', type='room_scene')
client.datasets.add_groups(staging['slug_version'], [rooms[0]['id'], rooms[1]['id']])
print(staging['slug_version'], '| group_count =', client.datasets.get(staging['slug_version'])['group_count'])

# A room_scene group cannot enter a garment_pair dataset - the type gate again.
try:
    client.datasets.add_groups(V2, [rooms[0]['id']])
    print('BUG: a garment_pair dataset accepted a room_scene group')
except DataRoomError as e:
    print('type mismatch rejected ->', why(e))

role exists empty_room


role exists staged_room
group_type exists room_scene


staging-scenes-b249d6/1 | group_count = 2
type mismatch rejected -> {"group_ids":"groups not of type garment_pair: 65bc52f0-2419-40f4-9375-e5ad6bf6e318"}


## 8. Cleanup

Everything above carries the run id. Frozen datasets delete fine - freezing gates membership
writes, not the dataset's own lifecycle.

In [11]:
for d in client.datasets.list(search=RUN):
    client.datasets.delete(d['slug_version'])
    print('deleted dataset', d['slug_version'])

stale = []
for g in client.groups.list(search=RUN, limit=100_000):
    stale.append(g['id'])
if stale:
    print('deleted groups:', client.groups.delete_many(stale)['deleted_count'])

deleted dataset staging-scenes-b249d6/1


deleted dataset tryon-eval-b249d6/1
deleted dataset tryon-pairs-b249d6/2


deleted dataset tryon-pairs-b249d6/1


deleted dataset tryon-train-b249d6/1


deleted groups: 8


## 9. Recap

| Use case | The calls that carry it |
|---|---|
| Paired training data | `groups.create_many`, `datasets.create`, `datasets.add_groups`, `datasets.freeze`, `datasets.groups(include_presigned_urls=True)` |
| Train/eval split | `datasets.create` x2, `datasets.add_groups`, `datasets.freeze` |
| The next drop | `datasets.copy` (same slug -> next version), `datasets.add_groups` on v2 only |
| What is still missing | `groups.list(dataset=..., has_role=[...])`, `images.list(datasets=[...], roles=[...])` |
| A second taxonomy | `roles.create`, `group_types.create`, the same dataset flow under new names |

For the per-call detail behind any of these, see [groups_example](groups_example.ipynb) and
[datasets_example](datasets_example.ipynb).